In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet torch

In [ ]:
!pip install --quiet pytorch-lightning
!pip install --quiet torchmetrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

mypath = '/content/drive/MyDrive/AttackDetectionML'

 # Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import time
from os.path import join as pjoin
from pandas import read_pickle as rpckl
from pandas import read_csv as rcsv
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, fbeta_score

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import FBetaScore

from multiprocessing import cpu_count
import random

import shutil

In [ ]:
random.seed(42)
pl.seed_everything(42)  # Reproduce thing

# Parameters

## Datasets

In [ ]:
net = 'CH'
ds_type = 'generation'
seq = 4

## Path

In [ ]:
data_folder = pjoin(mypath, 'raw_data')
ds_path = pjoin(mypath, 'datasets', net)
res_dir = pjoin(mypath, 'results', 'supervised', 'single_node_attack', net)

print(f'data_folder:\n\t{data_folder}\n')
print(f'ds_path:\n\t{ds_path}\n')
print(f'res_dir:\n\t{res_dir}\n')

## Model

In [ ]:
N_EPOCHS = 50
BATCH_SIZE = 128
DROPOUT = 0.75
LEARNING_RATE = 0.0001
N_HIDDEN = 256
N_LAYERS = 3

LOSS_WEIGHTS = torch.FloatTensor([0.1, 0.9])  # Imbalanced dataset

# Class

## Data Module

In [ ]:
class AnomalyDataModule(pl.LightningDataModule):

  def __init__(self, train_ds, val_ds, test_ds, batch_size):
    super().__init__()
    self.train_dataset = train_ds
    self.val_dataset = val_ds
    self.test_dataset = test_ds
    self.batch_size = batch_size

  def train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self.batch_size,
        shuffle=True,
        num_workers=cpu_count()
        )

  def val_dataloader(self):
    return DataLoader(
        self.val_dataset,
        batch_size=self.batch_size,
        shuffle=False,
        num_workers=cpu_count()
        )

  def test_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=self.batch_size,
        shuffle=False,
        num_workers=cpu_count()
        )

## Model

In [ ]:
class SequenceModel(nn.Module):
  def __init__(self, n_features, n_classes, DROPOUT,
               n_hidden=N_HIDDEN, n_layers=N_LAYERS):
    super().__init__()

    self.lstm = nn.LSTM(
        input_size=n_features,
        hidden_size=n_hidden,
        num_layers=n_layers,
        batch_first=True,
        dropout=DROPOUT,
    )

    self.classifier = nn.Linear(n_hidden, n_classes)

  def forward(self, x):
    self.lstm.flatten_parameters()
    _, (hidden, _) = self.lstm(x)

    out = hidden[-1]
    return self.classifier(out)

## Predictor

In [ ]:
class AnomalyPredictor(pl.LightningModule):

  def __init__(self, n_features: int, n_classes: int):
    super().__init__()
    self.model = SequenceModel(n_features, n_classes, DROPOUT)
    self.criterion = nn.CrossEntropyLoss(weight=LOSS_WEIGHTS)
    self.f2_scorer = FBetaScore(task="binary", beta=2.)

  def forward(self, x, labels=None):
    output = self.model(x)
    loss = 0
    if labels is not None:
      loss = self.criterion(output, labels)
    return loss, output

  def training_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    predictions = torch.argmax(outputs, dim=1)
    step_f2_score = self.f2_scorer(predictions, labels)

    self.log('train_loss', loss, prog_bar=True, logger=True)
    self.log('train_f2_score', step_f2_score, prog_bar=True, logger=True)

    return {'loss': loss, 'f2_score': step_f2_score}

  def validation_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    predictions = torch.argmax(outputs, dim=1)
    step_f2_score = self.f2_scorer(predictions, labels)

    self.log('val_loss', loss, prog_bar=True, logger=True)
    self.log('val_f2_score', step_f2_score, prog_bar=True, logger=True)
    return {'loss': loss, 'f2_score': step_f2_score}

  def test_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    predictions = torch.argmax(outputs, dim=1)
    step_f2_score = self.f2_scorer(predictions, labels)

    self.log('test_loss', loss, prog_bar=True, logger=True)
    self.log('test_f2_score', step_f2_score, prog_bar=True, logger=True)
    return {'loss': loss, 'f2_score': step_f2_score}

  def configure_optimizers(self):
    return optim.Adam(self.parameters(), lr=LEARNING_RATE)

# Load data


## Features

In [ ]:
def load_data(country_code, n_year=20):

    gen_info = rcsv(pjoin(data_folder, 'gens_info.csv'))
    gen_info = gen_info[gen_info.country==country_code]


    # DOWNLOAD GEN FIRST YEAR
    year = 2016
    index = 1
    print(f'>>> importing gen data: year {year}, series {index}')
    gen = rcsv(pjoin(data_folder, f'gens_{year}_{index}.csv'))
    gen.columns = gen.columns.astype(int)
    gen = gen[gen_info.id]
    gen *= 100  # per units /!\

    # DOWNLOAD GEN OTHER YEARS
    for i in range(n_year - 1):  # DOWNLOAD AND ADD N-1 OTHERS YEARS
        year += 1
        if year > 2020:
            index += 1
            year = 2016
        if index > 4:
            break

        print(f'>>> importing gen data: year {year}, series {index}')
        df = rcsv(pjoin(data_folder, f'gens_{year}_{index}.csv'))
        df.columns = df.columns.astype(int)
        df = df[gen_info.id]
        df *= 100  # per units /!\

        gen = pd.concat([gen, df], ignore_index=True)

    if ds_type == 'generation':
        load = []
    else:
        load_info = rcsv(pjoin(data_folder, 'loads_info.csv'))
        load_info = load_info[load_info.country==country_code]


        # DOWNLOAD LOAD FIRST YEAR
        year = 2016
        index = 1
        print(f'>>> importing load data: year {year}, series {index}')
        load = rcsv(pjoin(data_folder, f'loads_{year}_{index}.csv'))
        load.columns = load.columns.astype(int)
        load = load[load_info.id]
        load *= 100  # per units /!\

        # DOWNLOAD LOAD OTHER YEARS
        for i in range(n_year - 1):  # DOWNLOAD AND ADD N-1 OTHERS YEARS
            year += 1
            if year > 2020:
                index += 1
                year = 2016
            if index > 4:
                break

            print(f'>>> importing load data: year {year}, series {index}')
            df = rcsv(pjoin(data_folder, f'loads_{year}_{index}.csv'))
            df.columns = df.columns.astype(int)
            df = df[load_info.id]
            df *= 100

            load = pd.concat([load, df], ignore_index=True)

        # Add suffix to load to avoid duplicated in injection dataset
        load.columns = load.columns.astype(str)
        load = load.add_suffix('_load')

    return gen, load

In [ ]:
gen_p, load_p = load_data(net)

train_index = rpckl(pjoin(ds_path, 'regression_train_timesteps.p')).to_list()
val_index = rpckl(pjoin(ds_path, 'regression_validation_timesteps.p')).to_list()
test_index = rpckl(pjoin(ds_path, 'test_timesteps.p'))[0].to_list()

## Labels

In [ ]:
attacked_index = rpckl(pjoin(ds_path, 'attacked_timesteps.p'))[0].to_list()

y = pd.DataFrame({'class_name': 'normal'}, index=gen_p.index)
y.loc[attacked_index, 'class_name'] = 'anomaly'

## LABEL ENCODER
label_encoder = LabelEncoder()
label_encoder.fit(y.class_name)

# label 0 is normal, label 1 is anomaly
label_encoder.classes_ =np.array(['normal', 'anomaly'], dtype=object)

y['label'] = label_encoder.transform(y.class_name)

# Avoid printing unwanted logs

In [ ]:
# Remove prompt that appear every times

"""
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
WARNING:pytorch_lightning.loggers.tensorboard:Missing logger folder: lightning_logs/Sils_gen_generation
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
"""

import logging
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.loggers.tensorboard").setLevel(logging.ERROR)

# LOOP FOR ALL NODES


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

attacked_gens = rpckl(pjoin(ds_path, 'attacked_gens.p')).to_list()

start_time = time.time()  # For total running time
st = time.time()
for attacked_gen in attacked_gens:
  print(f'\n{attacked_gen} - {ds_type} - {time.time()-st:.1f} seconds')
  st = time.time()


  ## Path manager
  res_path = pjoin(res_dir, 'lstm', ds_type, f'{attacked_gen}')
  os.makedirs(res_path, exist_ok=True)



  ## LOAD DATASET
  X = gen_p.copy()

  if ds_type == 'injection':
      X = pd.concat([X, load_p], axis=1)

  # add anomalies
  attacked_gen_p = rpckl(pjoin(ds_path, f'{attacked_gen}_p_attacked.p'))
  X_attacked = X.copy()
  X_attacked[attacked_gen] = attacked_gen_p

  ## SCALER
  X.columns = X.columns.astype(str)
  X_attacked.columns = X_attacked.columns.astype(str)
  X_scaler = MinMaxScaler()
  X = X_scaler.fit_transform(X)
  X_attacked = X_scaler.transform(X_attacked)

  X = X.reshape((20, 8736, -1))

  # put together the features with their history
  X = np.expand_dims(X, 2)
  X = np.concatenate([np.roll(X, seq - t, axis=1) for t in range(seq)], axis=2)
  X = X.reshape((20*8736, seq, -1))
  X = np.concatenate([X, np.expand_dims(X_attacked, 1)], axis=1)

  # split dataset
  X_train = X[train_index]
  X_val = X[val_index]
  X_test = X[test_index]

  y_train = y['label'][train_index].values
  y_val = y['label'][val_index].values
  y_test = y['label'][test_index].values


  ## LOAD TRAIN, VAL & TEST DATASET
  train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long())
  val_ds = TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).long())
  test_ds = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long())

  ## Data Module
  data_module = AnomalyDataModule(train_ds, val_ds, test_ds, BATCH_SIZE)

  ## Model
  model = AnomalyPredictor(n_features = X_train.shape[-1],
                           n_classes = len(label_encoder.classes_))
  model.to(device)


  ## Checkpoint & Logger
  param_str = f'Seq-{seq}ts_Hid-{N_HIDDEN}_Lay-{N_LAYERS}'
  checkpoint_callback = ModelCheckpoint(
                            dirpath=f'checkpoints/{attacked_gen}/{ds_type}',
                            filename=f'best_{param_str}',
                            save_top_k=1,
                            verbose=False,
                            monitor='val_loss',
                            mode='min',
                                      )

  logs_file_name = f'{attacked_gen}_{ds_type[:3]}_{param_str}'
  logger = TensorBoardLogger('lightning_logs',
                             name=logs_file_name)


  ## Trainer
  trainer = pl.Trainer(
                logger=logger,
                callbacks=[checkpoint_callback],
                max_epochs=N_EPOCHS,
                accelerator='gpu',
                devices=1,
                enable_progress_bar=False,  # No progress bar
                enable_model_summary=False,  # No model summary
                      )


  ## Training
  trainer.fit(model, data_module)


  ## Save lightning_logs
  dst_folder = pjoin(mypath, 'supervised', 'lightning_logs',
                     net, logs_file_name)
  if os.path.exists(dst_folder): shutil.rmtree(dst_folder)
  shutil.copytree(
        pjoin('/content', 'lightning_logs', logs_file_name),
        dst_folder,)


  ## Save best model
  shutil.copy2(
      trainer.checkpoint_callback.best_model_path,
      pjoin(res_path, 'best_checkpoint.ckpt'),
      )


  ## Prediction
  trained_model = AnomalyPredictor.load_from_checkpoint(
                              trainer.checkpoint_callback.best_model_path,
                              n_features=X_train.shape[-1],
                              n_classes=len(label_encoder.classes_)
                                                      )
  trained_model.to(device)

  trained_model.freeze()  # just for inference, disable dropout and gradient calculation


  ## Test prediction
  _, output = trained_model(torch.tensor(X_test).float().to(device))
  y_test_predict = np.array(torch.argmax(output, dim=1).to('cpu'))


  ## Results
  n_train = len(y_train)
  hacked_train = y_train.sum()

  n_test = len(y_test)
  hacked_test = y_test.sum()

  f2_score = fbeta_score(y_test, y_test_predict,  beta=2)

  f5_score = fbeta_score(y_test, y_test_predict,  beta=5)

  cm = confusion_matrix(y_test, y_test_predict).ravel()
  cm = np.array(cm)

  confusion_df = pd.DataFrame({'train_size': 1,
                                'train_sample': n_train,
                                'train_hacked': hacked_train,
                                'train_occ': hacked_train/n_train,
                                'test_hacked': hacked_test,
                                'test_occ': hacked_test/n_test,
                                'f2_score': f2_score,
                                'f5_score': f5_score,
                                'tn': cm[0], 'fp': cm[1],
                                'fn': cm[2], 'tp': cm[3],
                                }, index=[0])


  ## Print results
  print(f'\tf2_score:\t{f2_score:.2f}')
  print(f'\tcm:\t{cm}\t[tn fp fn tp]')


  ## Save results
  confusion_df.to_csv(pjoin(res_path, 'confusion_df.csv'))  # CSV to avoid protocol problems


ex_time = int(time.time() - start_time)
print(f"\nTotal run time :\t{ex_time} [s]  -  {ex_time/3600:.2f} [h] ")

In [ ]:
from google.colab import runtime
runtime.unassign()  # disconnect and delete the runtime environment